# 01. Initial Data Exploration (`diabetes_data.csv`)

## Objective
Perform a thorough initial audit of `data/raw/diabetes_data.csv` **before any preprocessing or modeling** to understand:
1. Schema, column count, and data types.
2. Missing values and duplicate records.
3. Categorical vs. numerical variable groupings.
4. Distribution of the target variable `Diagnosis` (existing diabetes diagnosis).
5. Potential clinical anomalies, outliers, constant columns, and data leakage risks.

> **Framing Note:** `Diagnosis` represents an **existing cross-sectional diagnosis**, not future diabetes onset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes_data.csv"

df = pd.read_csv(RAW_DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

### Analysis: Dataset Dimensions
The raw dataset contains **1,879 patient records** and **46 columns**, covering demographics, lifestyle metrics, medical history, laboratory biomarkers, medications, symptoms, and environmental exposures. Next, we inspect exact data types, missing values, and duplicates.

In [ ]:
audit_df = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
print(f"Total missing cells: {df.isnull().sum().sum()}")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate PatientIDs: {df['PatientID'].duplicated().sum()}")
audit_df

### Analysis: Completeness, Duplicates, and Zero-Variance Fields
- **Missingness:** There are **0 missing values** across all 46 columns.
- **Duplicates:** There are **0 exact duplicate rows** and **1,879 unique `PatientID` values** (`6000`–`7878`).
- **Constant Column Alert:** `DoctorInCharge` has `n_unique = 1` (`"Confidential"` in every row), meaning it carries zero predictive or analytical variance and must be excluded prior to modeling.

In [ ]:
diag_counts = df["Diagnosis"].value_counts().sort_index()
diag_pct = (df["Diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2)
pd.DataFrame({"count": diag_counts, "percent": diag_pct})

### Analysis: Target Variable (`Diagnosis`)
The binary target `Diagnosis` is moderately imbalanced:
- `0` (No Diabetes): **1,127 patients (59.98%)**
- `1` (Diabetes): **752 patients (40.02%)**

Because of this ~60/40 split, evaluation should track **F1-score, Recall, Precision, ROC-AUC, and PR-AUC** alongside Accuracy, using **stratified** train/test splits.

In [ ]:
exclude_cols = {"PatientID", "DoctorInCharge", "Diagnosis"}
categorical_cols = [
    col for col in df.columns
    if col not in exclude_cols and df[col].nunique() <= 10
]
numerical_cols = [
    col for col in df.columns
    if col not in exclude_cols and col not in categorical_cols
]

print(f"Categorical/Binary features ({len(categorical_cols)}): {categorical_cols}")
print(f"Continuous/Numerical features ({len(numerical_cols)}): {numerical_cols}")
df[numerical_cols].describe().T[["min", "25%", "50%", "75%", "max", "mean", "std"]].round(3)

### Analysis: Categorical vs. Numerical Variables & Range Audit
Excluding `PatientID`, `DoctorInCharge`, and `Diagnosis`, the dataset contains **24 binary/categorical features** and **19 continuous/integer numerical features**. Next, we run domain-specific clinical consistency and outlier checks.

In [ ]:
bp_invalid = (df["DiastolicBP"] >= df["SystolicBP"]).sum()
gest_by_gender = df[df["GestationalDiabetes"] == 1]["Gender"].value_counts().to_dict()
pcos_by_gender = df[df["PolycysticOvarySyndrome"] == 1]["Gender"].value_counts().to_dict()
meds_crosstab = pd.crosstab(df["Diagnosis"], df["AntidiabeticMedications"], margins=True)

print(f"Rows with DiastolicBP >= SystolicBP: {bp_invalid} ({bp_invalid / len(df) * 100:.2f}%)")
print(f"GestationalDiabetes=1 by Gender: {gest_by_gender}")
print(f"PolycysticOvarySyndrome=1 by Gender: {pcos_by_gender}")
meds_crosstab

### Analysis: Clinical Data Quality & Leakage Findings
1. **Blood Pressure Ordering Anomaly:** **158 rows (8.41%)** have `DiastolicBP >= SystolicBP`, which is physiologically invalid and should be flagged during cleaning.
2. **Sex-Specific Condition Anomaly:** Both `GestationalDiabetes=1` (`Gender 0: 89, Gender 1: 98`) and `PolycysticOvarySyndrome=1` (`Gender 0: 42, Gender 1: 42`) occur across both gender codes.
3. **Treatment / Diagnostic Leakage Risk:** `AntidiabeticMedications`, `HbA1c`, and `FastingBloodSugar` reflect active treatment or direct diagnostic thresholds. Downstream modeling should test both full-feature classification and biomarker-excluded sensitivity runs.

## Summary & Next Steps
The raw dataset is complete (no missing cells or duplicate IDs), moderately imbalanced (40.02% positive `Diagnosis`), and contains documented clinical anomalies and a constant `DoctorInCharge` field. Proceed to `notebooks/02_data_cleaning.ipynb` and `src/data/preprocessing.py` to implement schema validation and leakage-safe preprocessing.